In [30]:
library(pacman)

p_load(shiny,
bslib,
leaflet,
dplyr,
lubridate,
ggplot2,
jsonlite)


In [31]:
# ==============================================================================
# 1. LOAD PRE-BUILT APP DATA
#    Run build_anim_data.R once (or after CSV changes) to generate:
#      data/app_data.rds   — R-side data for plots
#      www/anim_data.js    — static JS animation file (browser-cached)
# ==============================================================================

# Path-independent: works on any machine as long as the notebook is opened
# from the project root directory (standard Jupyter behaviour).
rds_path <- local({
  cands <- c(
    "data/app_data.rds",
    file.path(getwd(), "data", "app_data.rds"),
    file.path(dirname(getwd()), "data", "app_data.rds")
  )
  f <- Find(file.exists, cands)
  if (is.null(f))
    stop("app_data.rds not found. Run build_anim_data.R from the project directory first.")
  f
})

APP_DIR <- normalizePath(file.path(dirname(rds_path), ".."))
if (normalizePath(getwd()) != APP_DIR) {
  message("Setting working directory to: ", APP_DIR)
  setwd(APP_DIR)
}
# addResourcePath serves www/ at URL prefix "anim/", independent of getwd().
addResourcePath("anim", file.path(APP_DIR, "www"))

message("Loading app_data.rds...")
t0 <- proc.time()
.d <- readRDS(rds_path)

swift_clean     <- .d$swift_clean
swift_interp    <- .d$swift_interp
locs_snap       <- .d$locs_snap
sp_snap         <- .d$sp_snap
colony_colors   <- .d$colony_colors
unique_colonies <- .d$unique_colonies
unique_birds    <- .d$unique_birds
all_dates_vec   <- .d$all_dates_vec
rm(.d)

message("Loaded in ", round((proc.time()-t0)[["elapsed"]], 1), " s  |  ",
        length(unique_birds), " birds  |  ", length(all_dates_vec), " dates")

# ---- Derived objects -------------------------------------------------------
colony_palette <- colorFactor(
  palette = unname(colony_colors[unique_colonies]),
  domain  = unique_colonies)

SPERM_N <- 10L

# ---- Pre-compute annual migration cycle (DOY x colony mean latitude) -------
# CARTOGRAPHY NOTE: The annual cycle is a "temporal signature" of migration
# phenology (Slocum et al. Ch.20). Pre-computing outside the reactive graph
# avoids recomputation on every interaction. Loess smoothing (span=0.25)
# removes geolocator noise while preserving seasonal shape.
annual_cycle <- local({
  sc     <- swift_clean
  sc$doy <- as.integer(format(sc$timestamp, "%j"))
  agg    <- aggregate(lat ~ doy + colony, data = sc, FUN = mean, na.rm = TRUE)
  do.call(rbind, lapply(split(agg, agg$colony), function(x) {
    x <- x[order(x$doy), ]
    fit <- if (nrow(x) >= 10)
      tryCatch(loess(lat ~ doy, data = x, span = 0.25), error = function(e) NULL)
    else NULL
    x$smooth <- if (!is.null(fit)) pmax(-20, pmin(55, predict(fit))) else x$lat
    x
  }))
})

# ---- Shared dark ggplot2 theme ---------------------------------------------
# CARTOGRAPHY NOTE: A consistent dark background across map and plots preserves
# figure-ground hierarchy (Slocum Ch.9): the data (figure) remains the visual
# focus against the neutral dark ground, matching CartoDB.DarkMatter.
theme_swift <- function(base_size = 11) {
  ggplot2::theme_minimal(base_size = base_size) %+replace%
  ggplot2::theme(
    plot.background  = ggplot2::element_rect(fill = "#1e2130", colour = NA),
    panel.background = ggplot2::element_rect(fill = "#252836", colour = NA),
    text             = ggplot2::element_text(colour = "#c8cdd6"),
    axis.text        = ggplot2::element_text(colour = "#8a9099", size = 8),
    axis.title       = ggplot2::element_text(colour = "#c8cdd6", size = 9.5),
    panel.grid.major = ggplot2::element_line(colour = "#3a3f52", linewidth = 0.35),
    panel.grid.minor = ggplot2::element_blank(),
    legend.background= ggplot2::element_rect(fill = NA),
    legend.text      = ggplot2::element_text(colour = "#c8cdd6", size = 7.5),
    legend.key.size  = ggplot2::unit(0.38, "cm"),
    legend.key       = ggplot2::element_rect(fill = NA, colour = NA),
    plot.title       = ggplot2::element_text(colour = "#f0f2f5", face = "bold", size = 11),
    plot.subtitle    = ggplot2::element_text(colour = "#8a9099", size = 8.5,
                                              margin = ggplot2::margin(b = 6)),
    plot.margin      = ggplot2::margin(10, 12, 8, 10),
    axis.ticks       = ggplot2::element_line(colour = "#3a3f52")
  )
}

# ---- Legend HTML -----------------------------------------------------------
.build_legend_html <- function() {
  items <- paste(sapply(unique_colonies, function(col) {
    hex  <- colony_colors[[col]]
    safe <- paste0("lcb_", gsub("[^A-Za-z0-9]", "_", col))
    paste0(
      '<div style="display:flex;align-items:center;margin:3px 0;">',
        '<input type="checkbox" id="', safe, '" class="colony-leg-cb" value="', col, '" checked ',
               'style="margin:0 6px 0 0;cursor:pointer;" onchange="legendColonyChange()">',
        '<span style="background:', hex, ';border-radius:50%;display:inline-block;',
                     'width:10px;height:10px;margin-right:6px;flex-shrink:0;"></span>',
        '<label for="', safe, '" style="margin:0;font-size:11px;cursor:pointer;',
                        'white-space:nowrap;color:#dee2e6;">',
          col, '</label>',
      '</div>'
    )
  }), collapse = "\n")
  paste0(
    '<div class="colony-legend-box">',
      '<div style="font-weight:700;font-size:11px;margin-bottom:5px;color:#f8f9fa;">Colonies</div>',
      '<div style="margin-bottom:5px;">',
        '<button onclick="legSelAll()"  class="leg-btn leg-btn-primary">All</button>',
        '<button onclick="legSelNone()" class="leg-btn leg-btn-secondary">None</button>',
      '</div>', items, '</div>'
  )
}
LEGEND_HTML <- .build_legend_html()

# ---- Batched polyline builder (NaN-break trick) ----------------------------
make_colony_lines <- function(df) {
  parts <- lapply(unique(df$bird_id), function(b) {
    bd <- df[df$bird_id == b, ]
    if (nrow(bd) < 2) return(NULL)
    rbind(bd[, c("map_lon","map_lat")],
          data.frame(map_lon = NA_real_, map_lat = NA_real_))
  })
  parts <- Filter(Negate(is.null), parts)
  if (!length(parts)) return(NULL)
  do.call(rbind, parts)
}

Loading app_data.rds...
Loaded in 0.8 s  |  173 birds  |  1978 dates


In [32]:
# ==============================================================================
# 2. UI
# ==============================================================================
ui <- page_navbar(
  title    = tags$span(
    tags$span(style = "letter-spacing:.5px; font-weight:700;", "ALPINE SWIFT"),
    tags$span(style = "font-weight:300; opacity:.75;", " Migration Atlas")
  ),
  theme    = bs_theme(
    version    = 5,
    bootswatch = "darkly",
    primary    = "#4e79a7",
    success    = "#59a14f",
    info       = "#76b7b2",
    "font-size-base" = "0.84rem"
  ),
  fillable = FALSE,

  tags$head(
    tags$script(src = "anim/anim_data.js"),

    tags$style(HTML("
      .shiny-notification{position:fixed;top:16px;right:16px;bottom:auto;left:auto;z-index:9999;}

      /* Value boxes: compact with small text */
      .value-box .value-box-value{font-size:0.70rem!important;font-weight:600;}
      .value-box .value-box-title{font-size:0.60rem!important;opacity:.8;}

      /* Colony legend on map */
      .colony-legend-box{
        background:#1e2130;padding:7px 10px;border-radius:6px;
        box-shadow:0 2px 8px rgba(0,0,0,.55);
        max-height:310px;overflow-y:auto;min-width:155px;
      }
      .leg-btn{font-size:10px;padding:1px 7px;cursor:pointer;border-radius:3px;margin-right:3px;}
      .leg-btn-primary  {border:1px solid #4e79a7;color:#4e79a7;background:transparent;}
      .leg-btn-secondary{border:1px solid #6c757d;color:#6c757d;background:transparent;}

      /* Selected-bird badge */
      #sel_bird_ui .sel-badge{
        background:#252836;border:1px solid #3a3f52;border-radius:5px;
        padding:5px 8px;font-size:11px;line-height:1.4;color:#dee2e6;
      }
      #sel_bird_ui .sel-badge b{color:#f0f2f5;}

      /* Play-row */
      .play-row .form-group{margin-bottom:0!important;}
      .play-row .selectize-control{margin-bottom:0;}
      .play-row .selectize-input{
        min-height:29px!important;height:29px!important;
        line-height:20px!important;padding:3px 7px!important;
      }
      .play-row .btn{height:29px;line-height:1;}

      /* Leaflet layers control */
      .leaflet-control-layers{background:#1e2130!important;border:1px solid #3a3f52!important;color:#dee2e6!important;}
      .leaflet-control-layers label{color:#dee2e6!important;}

      /* Map wrapper for no-data overlay positioning */
      .map-wrap{position:relative;width:100%;height:100%;}
      #noDataOverlay{
        display:none;position:absolute;top:50%;left:50%;
        transform:translate(-50%,-50%);
        background:rgba(30,33,48,0.92);padding:10px 18px;border-radius:8px;
        color:#c8cdd6;font-size:12px;pointer-events:none;z-index:1000;
        border:1px solid #3a3f52;text-align:center;white-space:nowrap;
      }

      /* Hint strip below plot tab bar */
      .plot-hint{
        font-size:9px;color:#505870;padding:1px 10px 2px;
        display:flex;justify-content:space-between;align-items:center;
      }
      .plot-hint-zoom{color:#e8c46a;font-style:italic;}
    ")),

    tags$script(HTML("
      /* ---- Colony legend -> Shiny ---- */
      function legendColonyChange() {
        var v = [];
        document.querySelectorAll('.colony-leg-cb:checked')
          .forEach(function(cb){ v.push(cb.value); });
        Shiny.setInputValue('colony_filter', v, {priority:'event'});
      }
      function legSelAll()  {
        document.querySelectorAll('.colony-leg-cb').forEach(function(cb){ cb.checked=true;  });
        legendColonyChange();
      }
      function legSelNone() {
        document.querySelectorAll('.colony-leg-cb').forEach(function(cb){ cb.checked=false; });
        legendColonyChange();
      }

      /* ================================================================
         CLIENT-SIDE ANIMATION ENGINE
      ================================================================ */
      (function(){
        var D              = window._ANIM_DATA || null;
        var idx            = 0;
        var timer          = null;
        var pool           = {};
        var trailBuf       = {};
        var trailLines     = {};
        var renderer       = null;
        var highlightBird  = null;
        var activeColonies = null;
        var activeBirdFlt  = null;
        var TRAIL_N        = 12;
        var TRAIL_SEGS     = 5;
        var plotTmr        = null;
        var slideTmr       = null;

        function getMap() {
          var w = window.HTMLWidgets && HTMLWidgets.find && HTMLWidgets.find('#map');
          return w && w.getMap ? w.getMap() : null;
        }
        function getRenderer() {
          if (!renderer) renderer = L.canvas({ padding: 0.5 });
          return renderer;
        }

        function _shouldShow(bid, ci) {
          if (activeBirdFlt  && bid !== activeBirdFlt)          return false;
          if (activeColonies && D.colNames &&
              activeColonies.indexOf(D.colNames[ci]) === -1)    return false;
          return true;
        }

        function _markerStyle(bid, colIdx) {
          var fc = D.colors[colIdx];
          if (!highlightBird)
            return { radius:6,  fillColor:fc, fillOpacity:0.85, color:'#2a2a2a', weight:1.5 };
          if (bid === highlightBird)
            return { radius:10, fillColor:fc, fillOpacity:1.0,  color:'#FFD700', weight:2.5 };
          return   { radius:4,  fillColor:fc, fillOpacity:0.15, color:'#2a2a2a', weight:0.8 };
        }

        function _drawTrail(bid, colIdx, m) {
          if (trailLines[bid]) trailLines[bid].forEach(function(l){ l.remove(); });
          trailLines[bid] = [];
          var buf = trailBuf[bid];
          if (!buf || buf.length < 2) return;
          var n    = buf.length;
          var col  = D.colors[colIdx];
          var step = Math.max(1, Math.floor((n - 1) / TRAIL_SEGS));
          for (var s = 0; s < n - 1; s += step) {
            var frac = (s + step) / n;
            trailLines[bid].push(
              L.polyline([buf[s], buf[Math.min(s + step, n - 1)]], {
                color: col, weight: 1.0 + 3.2 * frac, opacity: 0.06 + 0.64 * frac
              }).addTo(m)
            );
          }
        }

        if (D) console.log('[jsAnim] ' + D.dates.length + ' frames');
        else   console.warn('[jsAnim] _ANIM_DATA missing — check anim/anim_data.js');

        Shiny.addCustomMessageHandler('jsAnimCmd', function(msg) {
          if      (msg.cmd === 'play')  _play(msg.ms, msg.startDate);
          else if (msg.cmd === 'pause') _pause();
          else if (msg.cmd === 'seek')  _seek(msg.date);
        });
        Shiny.addCustomMessageHandler('jsHighlight', function(msg) {
          highlightBird = msg.bird_id || null;
          _frame(idx);
        });
        Shiny.addCustomMessageHandler('jsFilter', function(msg) {
          activeColonies = msg.colonies || null;
          activeBirdFlt  = msg.bird     || null;
          _frame(idx);
        });
        Shiny.addCustomMessageHandler('showNoData', function(msg) {
          var od = document.getElementById('noDataOverlay');
          if (od) od.style.display = msg.show ? 'flex' : 'none';
        });
        /* R will call this ONLY when it has successfully drawn the new frame */
        Shiny.addCustomMessageHandler('clearJsAnim', function(msg) {
          _clearPool();
        });

        function _frame(i) {
          if (!D) return;
          var m = getMap(); if (!m) return;
          idx = i;
          var snap = D.snaps[i] || [], active = {}, visible = 0;
          for (var k = 0; k < snap.length; k++) {
            var p = snap[k], la = p[0], lo = p[1], ci = p[2], bid = p[3];
            if (!_shouldShow(bid, ci)) {
              if (pool[bid]) { pool[bid].remove(); delete pool[bid]; }
              if (trailLines[bid]) { trailLines[bid].forEach(function(l){ l.remove(); }); delete trailLines[bid]; }
              delete trailBuf[bid]; continue;
            }
            active[bid] = true; visible++;
            if (!trailBuf[bid]) trailBuf[bid] = [];
            trailBuf[bid].push([la, lo]);
            if (trailBuf[bid].length > TRAIL_N) trailBuf[bid].shift();
            var st = _markerStyle(bid, ci);
            if (pool[bid]) {
              pool[bid].setLatLng([la, lo]);
              pool[bid].setStyle({ fillColor:st.fillColor, fillOpacity:st.fillOpacity, color:st.color, weight:st.weight });
              pool[bid].setRadius(st.radius);
            } else {
              pool[bid] = L.circleMarker([la, lo], {
                renderer: getRenderer(), radius:st.radius,
                color:st.color, weight:st.weight, fillColor:st.fillColor, fillOpacity:st.fillOpacity
              }).addTo(m);
              (function(id) {
                pool[id].on('click', function(e) {
                  L.DomEvent.stopPropagation(e);
                  Shiny.setInputValue('map_marker_click', {id:id}, {priority:'event'});
                });
              })(bid);
            }
            _drawTrail(bid, ci, m);
          }
          Object.keys(pool).forEach(function(bid) {
            if (!active[bid]) {
              pool[bid].remove(); delete pool[bid];
              if (trailLines[bid]) { trailLines[bid].forEach(function(l){ l.remove(); }); delete trailLines[bid]; }
              delete trailBuf[bid];
            }
          });
          var od = document.getElementById('noDataOverlay');
          if (od) od.style.display = visible === 0 ? 'flex' : 'none';
          clearTimeout(plotTmr);
          plotTmr = setTimeout(function() {
            Shiny.setInputValue('js_plot_date', D.dates[i], {priority:'event'});
          }, 50);
        }

        function _clearPool() {
          Object.keys(pool).forEach(function(bid) { pool[bid].remove(); });
          Object.keys(trailLines).forEach(function(bid) { trailLines[bid].forEach(function(l){ l.remove(); }); });
          pool = {}; trailLines = {}; trailBuf = {};
        }

        function _play(ms, startDate) {
          if (!D) D = window._ANIM_DATA || null;
          if (!D) { console.error('[jsAnim] _ANIM_DATA null — 404?'); return; }
          if (startDate) { var si = D.dates.indexOf(startDate); if (si >= 0) idx = si; }
          if (timer) clearInterval(timer);
          _frame(idx);
          timer = setInterval(function() {
            if (idx + 1 >= D.dates.length) { _pause(); return; }
            idx = idx + 1; _frame(idx);
          }, ms || 400);
          Shiny.setInputValue('js_anim_active', true, {priority:'event'});
        }

        function _pause() {
          if (timer) { clearInterval(timer); timer = null; }
          
          /* Do NOT clear the pool here! Let the markers stay until R takes over */
          
          Shiny.setInputValue('js_anim_active', false, {priority:'event'});
          if (D) Shiny.setInputValue('js_frame_date', D.dates[idx], {priority:'event'});
        }

        function _seek(dateStr) {
          if (!D) return;
          var i = D.dates.indexOf(dateStr);
          if (i >= 0) { idx = i; _frame(i); }
        }

        window._jsAnim = { play:_play, pause:_pause, seek:_seek };
      })();
    "))
  ),

  sidebar = sidebar(
    width = 250,
    bg = "#1e2130",

    h6(class = "text-uppercase text-muted fw-bold mb-1",
       style = "font-size:9.5px;letter-spacing:.8px;", "Individual"),
    selectInput("bird_sel", NULL, choices = c("All Birds", unique_birds), width = "100%"),

    hr(class = "my-2", style = "border-color:#3a3f52;"),
    h6(class = "text-uppercase text-muted fw-bold mb-1",
       style = "font-size:9.5px;letter-spacing:.8px;", "Timeline"),
    uiOutput("slider_ui"),
    div(
      class = "d-flex align-items-center gap-2 mt-2 play-row",
      actionButton("btn_play",  HTML(as.character(bsicons::bs_icon("play-fill"))),
                   class = "btn btn-sm btn-success", title = "Play"),
      actionButton("btn_pause", HTML(as.character(bsicons::bs_icon("pause-fill"))),
                   class = "btn btn-sm btn-secondary", title = "Pause"),
      selectInput("play_speed", NULL,
                  choices  = c("8x"=50,"4x"=100,"2x"=200,"1x"=400),
                  selected = 400, width = "70px")
    ),

    hr(class = "my-2", style = "border-color:#3a3f52;"),
    h6(class = "text-uppercase text-muted fw-bold mb-1",
       style = "font-size:9.5px;letter-spacing:.8px;", "Overlays"),
    checkboxInput("opt_sperm", "Moving point trails (last 10 fixes)", value = TRUE),

    uiOutput("sel_bird_ui"),

    tags$p(class = "text-muted mt-2 mb-0", style = "font-size:9.5px; line-height:1.5;",
      em("Map legend: toggle colonies on/off"), tags$br(),
      em("Click bird to highlight & view stats"), tags$br(),
      em("Click map background to deselect"), tags$br(),
      em("Plots auto-zoom to selected bird"), tags$br(),
      em("Data: Movebank / Meier et al. 2020"))
  ),

  nav_panel(
    title = "Map & Analytics",

    # ── Row 1: full-width map ───────────────────────────────────────────────
    card(
      full_screen = TRUE,
      height = "420px",
      style  = "background:#1e2130; border:1px solid #3a3f52;",
      card_header(
        style = "background:#252836; border-bottom:1px solid #3a3f52; padding:5px 12px;",
        tags$span(bsicons::bs_icon("geo-alt-fill"), " Live Migration Map")
      ),
      div(class = "map-wrap",
        leafletOutput("map", width = "100%", height = "100%"),
        div(id = "noDataOverlay",
            bsicons::bs_icon("calendar-x"),
            HTML("&nbsp;Breeding season in Europe &mdash; no migration data"))
      ),
      card_footer(
        class = "text-muted",
        style = "background:#252836; border-top:1px solid #3a3f52; font-size:9.5px; padding:4px 10px;",
        "Positions: midpoint of geolocator interval + per-bird jitter. ",
        "Layers control (top-right) switches basemap."
      )
    ),

    # # ── Row 2: value boxes ─────────────────────────────────────────────────
    # layout_columns(
    #   col_widths = c(4, 4, 4), gap = "0.5rem",
    #   style = "margin-top:0.5rem;",
    #   value_box("GPS Fixes",    textOutput("stat_fixes"),
    #             theme = value_box_theme(bg="#252836", fg="#c8cdd6"), height="58px"),
    #   value_box("Active Birds", textOutput("stat_birds"),
    #             theme = value_box_theme(bg="#252836", fg="#c8cdd6"), height="58px"),
    #   value_box("Avg Latitude", textOutput("stat_lat"),
    #             theme = value_box_theme(bg="#252836", fg="#c8cdd6"), height="58px")
    # ),

    # ── Row 3: linked analytics ────────────────────────────────────────────
    navset_card_underline(
      title = div(
        style = "display:flex;align-items:center;gap:8px;",
        "Linked Analytics",
        uiOutput("plot_zoom_badge", inline = TRUE)
      ),
      nav_panel("Migration Profile",
                div(class = "plot-hint",
                    span("60-day trailing window"),
                    uiOutput("hint_profile", inline = TRUE)),
                plotOutput("plt_profile", height = "200px",
                           click    = clickOpts(id = "plt_profile_click"),
                           brush    = brushOpts(id = "plt_profile_brush", resetOnNew = TRUE),
                           dblclick = dblclickOpts(id = "plt_profile_dbl"))),
                           
      nav_panel("Latitude Distribution",
                div(class = "plot-hint",
                    span("Current date snapshot"),
                    uiOutput("hint_boxplot", inline = TRUE)),
                plotOutput("plt_boxplot", height = "200px",
                           brush    = brushOpts(id = "plt_boxplot_brush", resetOnNew = TRUE),
                           dblclick = dblclickOpts(id = "plt_boxplot_dbl"))),
                           
      nav_panel("Annual Cycle",
                div(class = "plot-hint",
                    span("Loess-smoothed by day-of-year"),
                    uiOutput("hint_cycle", inline = TRUE)),
                plotOutput("plt_cycle", height = "200px",
                           brush    = brushOpts(id = "plt_cycle_brush", resetOnNew = TRUE),
                           dblclick = dblclickOpts(id = "plt_cycle_dbl"))
      ))
  )
)

Warning message:
Navigation containers expect a collection of `bslib::nav_panel()`/`shiny::tabPanel()`s and/or `bslib::nav_menu()`/`shiny::navbarMenu()`s. Consider using `header` or `footer` if you wish to place content above (or below) every panel's contents. 


In [33]:
# ==============================================================================
# 3. SERVER
# ==============================================================================
server <- function(input, output, session) {

  # =========================================================================
  # A. SELECTION STATE
  # =========================================================================
  selected_bird <- reactiveVal(NULL)

  active_bird <- reactive({ selected_bird() })
  stats_bird  <- reactive({ selected_bird() })

  # =========================================================================
  # ZOOM STATES FOR PLOTS
  # =========================================================================
  zoom_profile <- reactiveValues(x = NULL, y = NULL)
  zoom_boxplot <- reactiveValues(x = NULL, y = NULL)
  zoom_cycle   <- reactiveValues(x = NULL, y = NULL)

  # Profile Zoom
  observeEvent(input$plt_profile_brush, {
    zoom_profile$x <- as.Date(c(input$plt_profile_brush$xmin, input$plt_profile_brush$xmax), origin="1970-01-01")
    zoom_profile$y <- c(input$plt_profile_brush$ymin, input$plt_profile_brush$ymax)
  })
  observeEvent(input$plt_profile_dbl, { zoom_profile$x <- NULL; zoom_profile$y <- NULL })

  # Boxplot Zoom
  observeEvent(input$plt_boxplot_brush, {
    zoom_boxplot$x <- c(input$plt_boxplot_brush$xmin, input$plt_boxplot_brush$xmax)
    zoom_boxplot$y <- c(input$plt_boxplot_brush$ymin, input$plt_boxplot_brush$ymax)
  })
  observeEvent(input$plt_boxplot_dbl, { zoom_boxplot$x <- NULL; zoom_boxplot$y <- NULL })

  # Cycle Zoom
  observeEvent(input$plt_cycle_brush, {
    zoom_cycle$x <- c(input$plt_cycle_brush$xmin, input$plt_cycle_brush$xmax)
    zoom_cycle$y <- c(input$plt_cycle_brush$ymin, input$plt_cycle_brush$ymax)
  })
  observeEvent(input$plt_cycle_dbl, { zoom_cycle$x <- NULL; zoom_cycle$y <- NULL })

  # Per-bird latitude range pre-computed once for auto-zoom
  bird_lat_range <- local({
    lapply(split(swift_clean, swift_clean$bird_id), function(bd) {
      lats <- bd$lat[!is.na(bd$lat)]
      if (!length(lats)) return(NULL)
      pad <- max(8, diff(range(lats)) * 0.25)
      c(min(lats) - pad, max(lats) + pad)
    })
  })

  bird_ylim <- reactive({
    ab <- active_bird(); if (is.null(ab)) return(NULL)
    r  <- bird_lat_range[[ab]]; if (is.null(r)) return(NULL)
    c(max(r[1], -25), min(r[2], 60))
  })

  # ---- Map marker click ---------------------------------------------------
  observeEvent(input$map_marker_click, {
    bid     <- input$map_marker_click$id
    new_sel <- if (!is.null(selected_bird()) && selected_bird() == bid) NULL else bid
    selected_bird(new_sel)
    session$sendCustomMessage("jsHighlight", list(bird_id = new_sel))
  })

  # ---- Map background click: deselect ------------------------------------
  observeEvent(input$map_click, {
    selected_bird(NULL)
    session$sendCustomMessage("jsHighlight", list(bird_id = NULL))
  })

  # ---- Migration Profile click: select nearest bird ----------------------
  observeEvent(input$plt_profile_click, {
    click <- input$plt_profile_click; req(click)
    pb    <- plot_base(); ts <- as.Date(plot_ts())
    cur   <- pb[pb$timestamp >= ts - 60 & pb$timestamp <= ts, ]
    if (!nrow(cur)) return()
    click_date <- tryCatch(as.Date(click$x, origin = "1970-01-01"), error = function(e) NULL)
    req(!is.null(click_date))
    cn <- as.numeric(click_date)
    candidates <- do.call(rbind, lapply(split(cur, cur$bird_id), function(bd) {
      bd  <- bd[order(bd$timestamp), ]
      bn  <- as.numeric(bd$timestamp)
      if (cn < min(bn) || cn > max(bn)) return(NULL)
      lat_hat <- approx(bn, bd$plot_lat, xout = cn)$y
      if (is.na(lat_hat)) return(NULL)
      data.frame(bird_id = bd$bird_id[1], dist = abs(lat_hat - click$y))
    }))
    if (is.null(candidates) || !nrow(candidates)) return()
    best <- candidates[which.min(candidates$dist), ]
    if (best$dist > 4) return()
    bid     <- best$bird_id
    new_sel <- if (!is.null(selected_bird()) && selected_bird() == bid) NULL else bid
    selected_bird(new_sel)
    session$sendCustomMessage("jsHighlight", list(bird_id = new_sel))
  })

  # ---- Clear button -------------------------------------------------------
  observeEvent(input$btn_clear_sel, {
    selected_bird(NULL)
    session$sendCustomMessage("jsHighlight", list(bird_id = NULL))
  })

  # ---- Selected-bird badge ------------------------------------------------
  output$sel_bird_ui <- renderUI({
    sb <- selected_bird(); if (is.null(sb)) return(NULL)
    col <- swift_clean$colony[swift_clean$bird_id == sb][1]
    hex <- if (!is.null(col) && !is.na(col)) colony_colors[[col]] else "#888"
    tagList(
      h6(class = "text-uppercase text-muted fw-bold mb-1",
         style = "font-size:9.5px;letter-spacing:.8px;", "Selected"),
      div(class = "sel-badge",
        tags$span(style = paste0("display:inline-block;width:9px;height:9px;",
                                 "border-radius:50%;background:", hex, ";margin-right:5px;")),
        tags$b(sb), tags$br(),
        tags$span(style = "color:#8a9099;", col)
      ),
      actionButton("btn_clear_sel", "Clear",
                   class = "btn btn-sm btn-outline-secondary mt-2 px-2 py-1",
                   style = "font-size:11px;"),
      hr(class = "my-2", style = "border-color:#3a3f52;")
    )
  })

  output$plot_zoom_badge <- renderUI({
    if (is.null(active_bird())) return(NULL)
    tags$span(style = paste0("font-size:9px;color:#e8c46a;font-style:italic;",
                              "border:1px solid #5a4a20;background:#2a2510;",
                              "padding:1px 7px;border-radius:3px;"),
              "zoomed to bird")
  })
  output$hint_profile <- renderUI({
    if (!is.null(active_bird())) span(class="plot-hint-zoom", active_bird())
  })
  output$hint_boxplot <- renderUI({
    if (!is.null(active_bird())) span(class="plot-hint-zoom", active_bird())
  })
  output$hint_cycle <- renderUI({
    if (!is.null(active_bird())) span(class="plot-hint-zoom", active_bird())
  })

  # =========================================================================
  # B. ANIMATION CONTROLS
  # =========================================================================
  observeEvent(input$btn_play, {
    session$sendCustomMessage("jsAnimCmd", list(
      cmd       = "play",
      ms        = as.numeric(input$play_speed),
      startDate = as.character(input$time_slider)
    ))
  })
  observeEvent(input$btn_pause, {
    session$sendCustomMessage("jsAnimCmd", list(cmd = "pause"))
  })
  observeEvent(input$js_anim_active, {
    if (isTRUE(input$js_anim_active)) leafletProxy("map") %>% clearMarkers()
  }, ignoreInit = TRUE)

  # Immediately clear trails when toggled off
  observeEvent(input$opt_sperm, {
    if (!isTRUE(input$opt_sperm)) leafletProxy("map") %>% clearGroup("sperm")
  }, ignoreInit = TRUE)

  # Update slider safely only when paused
  observeEvent(input$js_frame_date, {
    req(input$js_frame_date)
    safe_time <- as.POSIXct(input$js_frame_date, tz="UTC")
    updateSliderInput(session, "time_slider", value = safe_time)
  }, ignoreInit = TRUE)

  # =========================================================================
  # C. FILTER SYNC TO JS DURING ANIMATION
  # =========================================================================
  observe({
    req(isTRUE(input$js_anim_active))
    cols <- active_cols_d(); sel <- input$bird_sel
    session$sendCustomMessage("jsFilter", list(
      colonies = as.list(cols),
      bird     = if (!is.null(sel) && nzchar(sel) && sel != "All Birds") sel else NULL
    ))
  })

  # =========================================================================
  # D. DATA REACTIVES
  # =========================================================================
  active_cols <- reactive({
    cf <- input$colony_filter
    if (is.null(cf) || !length(cf)) unique_colonies else cf
  })
  active_cols_d <- debounce(active_cols, 250)

  base_data <- reactive({
    d   <- swift_clean[swift_clean$colony %in% active_cols_d(), , drop = FALSE]
    sel <- input$bird_sel
    if (!is.null(sel) && sel != "All Birds") d <- d[d$bird_id == sel, , drop = FALSE]
    d
  })

  interp_data <- reactive({
    d   <- swift_interp[swift_interp$colony %in% active_cols_d(), , drop = FALSE]
    sel <- input$bird_sel
    if (!is.null(sel) && sel != "All Birds") d <- d[d$bird_id == sel, , drop = FALSE]
    d
  })

  plot_base <- reactive({
    d <- base_data(); d <- d[order(d$bird_id, d$timestamp), ]
    do.call(rbind, lapply(split(d, d$bird_id), function(bd) {
      bd$td  <- c(NA, as.numeric(diff(bd$timestamp)))
      bd$gap <- as.integer(is.na(bd$td) | bd$td > 15)
      bd$seg <- cumsum(bd$gap)
      bd$grp <- paste(bd$bird_id, bd$seg, sep = "_")
      bd
    }))
  })

  summer_gaps <- reactive({
    d <- base_data(); if (nrow(d) < 2) return(NULL)
    vd <- sort(unique(d$timestamp)); if (length(vd) < 2) return(NULL)
    df_val <- as.numeric(diff(vd)); s <- vd[-length(vd)]; e <- vd[-1]
    mask <- df_val > 15 & as.integer(format(s, "%m")) %in% 5:8
    if (!any(mask)) return(NULL)
    data.frame(s = s[mask], e = e[mask])
  })

  plot_ts <- debounce(reactive({
    pjd <- input$js_plot_date
    if (!is.null(pjd) && nzchar(pjd)) {
      as.POSIXct(pjd, tz="UTC")
    } else {
      req(input$time_slider)
    }
  }), 200)

  # =========================================================================
  # E. SLIDER
  # =========================================================================
  output$slider_ui <- renderUI({
    d <- base_data(); if (!nrow(d)) return(NULL)
    mn  <- min(d$timestamp, na.rm = TRUE)
    mx  <- max(d$timestamp, na.rm = TRUE)
    cur <- isolate(input$time_slider)
    if (is.null(cur) || cur < mn || cur > mx) cur <- mn
    sliderInput("time_slider", "Timeline", 
            min = min(all_dates_vec), 
            max = max(all_dates_vec), 
            value = min(all_dates_vec), 
            step = 3600 * 12,           
            timeFormat = "%Y-%m-%d %H:%M") 
  })

  # =========================================================================
  # F. MAP
  # =========================================================================
  output$map <- renderLeaflet({
    leaflet(options = leafletOptions(minZoom = 2)) %>%
      addProviderTiles(providers$CartoDB.DarkMatter,  group = "Dark")      %>%
      addProviderTiles(providers$CartoDB.Positron,    group = "Light")     %>%
      addProviderTiles(providers$Esri.WorldImagery,   group = "Satellite") %>%
      addLayersControl(baseGroups = c("Dark","Light","Satellite"),
                       options    = layersControlOptions(collapsed = TRUE)) %>%
      setView(lng = 10, lat = 20, zoom = 3) %>%
      addControl(html = LEGEND_HTML, position = "bottomright")
  })

  map_payload <- reactive({
    req(input$time_slider)
    ts <- input$time_slider
    
    idx <- which.min(abs(difftime(all_dates_vec, ts, units = "secs")))
    d_str <- as.character(all_dates_vec[idx])
    
    cols  <- active_cols_d(); sel <- input$bird_sel
    .filt <- function(df) {
      if (is.null(df) || !is.data.frame(df) || nrow(df) == 0 ||
          !"colony" %in% names(df)) return(NULL)
      df <- df[df$colony %in% cols, , drop = FALSE]
      if (!is.null(sel) && nzchar(sel) && sel != "All Birds")
        df <- df[df$bird_id == sel, , drop = FALSE]
      if (nrow(df) == 0) return(NULL); df
    }
    
    locs <- .filt(locs_snap[[d_str]])
    sp   <- if (isTRUE(input$opt_sperm)) .filt(sp_snap[[d_str]]) else NULL
    list(ts = ts, locs = locs, sp = sp)
  }) %>% debounce(250)


  # THIS IS THE BLOCK YOU ACCIDENTALLY DELETED EARLIER!
  observe({
    if (isTRUE(input$js_anim_active)) return()
    p <- map_payload(); req(p)
    
    no_data <- is.null(p$locs) || nrow(p$locs) == 0
    session$sendCustomMessage("showNoData", list(show = no_data))
    
    ab    <- active_bird()
    proxy <- leafletProxy("map") %>% clearGroup("sperm")
    
    if (!is.null(p$sp) && nrow(p$sp) > 0) {
      for (col in unique(p$sp$colony)) {
        sp_col      <- p$sp[p$sp$colony == col, ]
        bird_ids_sp <- unique(sp_col$bird_id)
        for (seg_i in seq_len(SPERM_N - 1L)) {
          frac  <- seg_i / SPERM_N
          parts <- lapply(bird_ids_sp, function(b) {
            bd <- sp_col[sp_col$bird_id == b, ]; bd <- bd[order(bd$timestamp), ]
            if (nrow(bd) <= seg_i) return(NULL)
            data.frame(map_lon = c(bd$map_lon[seg_i], bd$map_lon[seg_i+1L], NA_real_),
                       map_lat = c(bd$map_lat[seg_i], bd$map_lat[seg_i+1L], NA_real_))
          })
          parts <- Filter(Negate(is.null), parts); if (!length(parts)) next
          mat     <- do.call(rbind, parts)
          op_mult <- if (is.null(ab)) 1.0 else 0.2
          proxy <- proxy %>%
            addPolylines(lng = mat$map_lon, lat = mat$map_lat,
                         color   = colony_colors[col],
                         weight  = 1.2 + 4.0 * frac,
                         opacity = (0.12 + 0.70 * frac) * op_mult,
                         group   = "sperm")
        }
      }
    }
    
    if (!is.null(p$locs) && nrow(p$locs) > 0) {
      ld     <- p$locs
      is_sel <- if (!is.null(ab)) ld$bird_id == ab else rep(FALSE, nrow(ld))
      ld$rad  <- ifelse(is_sel, 10, 6)
      ld$scol <- ifelse(is_sel, "#FFD700", "#2a2a2a")
      ld$swt  <- ifelse(is_sel, 2.5, 1.5)
      ld$fop  <- if (!is.null(ab)) ifelse(is_sel, 1.0, 0.15) else 0.85
      proxy <- proxy %>%
        addCircleMarkers(data = ld, lng = ~map_lon, lat = ~map_lat,
                         layerId = ~bird_id, radius = ~rad,
                         color = ~scol, weight = ~swt,
                         fillColor = ~colony_palette(colony), fillOpacity = ~fop,
                         popup = ~paste0("<b>Colony:</b> ", colony, "<br>",
                                         "<b>Bird:</b> ",   bird_id, "<br>",
                                         "<b>Date:</b> ",   timestamp, "<br>",
                                         "<b>Lat:</b> ",    round(lat,2), "°N  ",
                                         "<b>Lon:</b> ",    round(lon,2), "°E"))
      gone <- setdiff(unique(base_data()$bird_id), ld$bird_id)
      if (length(gone)) proxy <- proxy %>% removeMarker(gone)
    } else {
      proxy <- proxy %>% removeMarker(unique(base_data()$bird_id))
    }
    
    # TELL JS TO CLEAR FROZEN MARKERS NOW THAT R HAS DRAWN REAL ONES
    session$sendCustomMessage("clearJsAnim", list())
  })

  # =========================================================================
  # G. STATS & PLOT DATA
  # =========================================================================
  locs_reactive <- reactive({
    p <- map_payload()
    df <- p$locs
    if (is.null(df) || nrow(df) == 0) return(NULL)
    
    sel <- input$bird_sel
    sb  <- stats_bird()
    
    if (!is.null(sb)) {
      df <- df[df$bird_id == sb, , drop = FALSE]
    } else if (!is.null(sel) && nzchar(sel) && sel != "All Birds") {
      df <- df[df$bird_id == sel, , drop = FALSE]
    }
    df
  })

  # =========================================================================
  # H. MIGRATION PROFILE PLOT
  # =========================================================================
  output$plt_profile <- renderPlot({
    ts  <- as.Date(plot_ts()); req(ts)    
    pb  <- plot_base()
    cur <- pb[pb$timestamp >= ts - 60L & pb$timestamp <= ts, ]
    if (!nrow(cur)) return(NULL)
    ab   <- active_bird()
    ylim <- bird_ylim()

    seg_n    <- ave(seq_len(nrow(cur)), cur$grp, FUN = length)
    cur_line <- cur[seg_n > 1, ]
    tips     <- do.call(rbind, lapply(split(cur, cur$bird_id),
                                      function(x) x[nrow(x), , drop = FALSE]))
    ba <- if (!is.null(ab)) 0.10 else 0.50
    bw <- if (!is.null(ab)) 0.35 else 0.65

    y_label_y <- if (!is.null(ylim)) ylim[1] + diff(ylim) * 0.04 else -13

    g <- ggplot(cur, aes(timestamp, plot_lat, colour = colony, group = grp)) +
      geom_line(data = cur_line, alpha = ba, linewidth = bw) +
      geom_point(data = tips, size = 1.5, alpha = ba) +
      geom_vline(xintercept = as.numeric(ts), colour = "#e8c46a",
                 linewidth = 0.85, linetype = "dashed") +
      annotate("text", x = ts + 2L, y = y_label_y,
               label = format(ts, "%d %b %Y"), colour = "#e8c46a", size = 2.9, hjust = 0)

    if (!is.null(ab) && ab %in% cur$bird_id) {
      hb_cur  <- cur[cur$bird_id == ab, ]
      hb_line <- hb_cur[ave(seq_len(nrow(hb_cur)), hb_cur$grp, FUN=length) > 1, ]
      hb_tip  <- tips[tips$bird_id == ab, ]
      g <- g +
        geom_line(data = hb_line, aes(group = grp), linewidth = 2.2, alpha = 1) +
        geom_point(data = hb_tip, size = 5.5, shape = 21,
                   fill = "#FFD700", colour = "#2a2a2a", stroke = 2)
    }
    g +
      scale_colour_manual(values = colony_colors) +
      scale_x_date(limits = ts + c(-60, 15), date_labels = "%b %Y") +
      scale_y_continuous(breaks = seq(-20, 60, 10),
                         labels = function(x) paste0(x, "°N")) +
      coord_cartesian(xlim = zoom_profile$x, ylim = if (!is.null(zoom_profile$y)) zoom_profile$y else if (!is.null(ylim)) ylim else c(-15, 55)) +
      theme_swift() +
      labs(x = NULL, y = "Latitude", colour = NULL,
           title = if (!is.null(ab)) paste0("Profile — ", ab)
                   else "Migration Profile (60-day window)") +
      theme(legend.position = "none")
  }, res = 110, bg = "#1e2130")

  # =========================================================================
  # I. LATITUDE DISTRIBUTION PLOT
  # =========================================================================
  output$plt_boxplot <- renderPlot({
    d  <- locs_reactive(); if (is.null(d) || !nrow(d)) return(NULL)
    ts <- as.Date(plot_ts())
    ab <- active_bird(); ylim <- bird_ylim()
    med_order <- tapply(d$map_lat, d$colony, median, na.rm = TRUE)
    col_order <- names(sort(med_order))
    d$col_ord <- factor(d$colony, levels = col_order)
    g <- ggplot(d, aes(col_ord, map_lat, fill = colony)) +
      geom_boxplot(alpha = if (!is.null(ab)) 0.20 else 0.80,
                   outlier.shape = 21, outlier.size = 1.2, width = 0.55, colour = "#555") +
      scale_fill_manual(values = colony_colors) +
      scale_y_continuous(breaks = seq(-20, 60, 10),
                         labels = function(x) paste0(x, "°N")) +
      coord_cartesian(xlim = zoom_boxplot$x, ylim = if (!is.null(zoom_boxplot$y)) zoom_boxplot$y else if (!is.null(ylim)) ylim else c(-15, 55)) +
      theme_swift() +
      labs(x = NULL, y = "Latitude",
           title = if (!is.null(ab)) paste0("Distribution — ", ab)
                   else paste0("Colony Latitude — ", format(ts, "%d %b %Y"))) +
      theme(legend.position = "none",
            axis.text.x = element_text(angle = 32, hjust = 1, size = 7.5))
    if (!is.null(ab) && ab %in% d$bird_id) {
      hb_row <- d[d$bird_id == ab, ]
      hb_row$col_ord <- factor(hb_row$colony, levels = col_order)
      g <- g + geom_point(data = hb_row, aes(x = col_ord, y = map_lat),
                          size = 5, shape = 21, fill = "#FFD700", colour = "#2a2a2a",
                          stroke = 2, inherit.aes = FALSE)
    }
    g
  }, res = 110, bg = "#1e2130")

  # =========================================================================
  # J. ANNUAL MIGRATION CYCLE PLOT
  # =========================================================================
  output$plt_cycle <- renderPlot({
    ac  <- annual_cycle[annual_cycle$colony %in% active_cols_d(), ]
    if (!nrow(ac)) return(NULL)
    ts      <- as.Date(plot_ts())
    ab      <- active_bird(); ylim <- bird_ylim()
    cur_doy <- as.integer(format(ts, "%j"))
    ba <- if (!is.null(ab)) 0.18 else 0.90
    bw <- if (!is.null(ab)) 0.5  else 1.1
    y_label_y <- if (!is.null(ylim)) ylim[1] + diff(ylim) * 0.04 else -19
    g <- ggplot(ac, aes(doy, smooth, colour = colony, group = colony)) +
      geom_line(alpha = ba, linewidth = bw) +
      geom_vline(xintercept = cur_doy, colour = "#e8c46a",
                 linewidth = 0.85, linetype = "dashed") +
      annotate("text", x = min(cur_doy + 5L, 355L), y = y_label_y,
               label = format(ts, "%d %b"), colour = "#e8c46a", size = 2.8, hjust = 0)
    if (!is.null(ab) && ab %in% swift_clean$bird_id) {
      bd <- swift_clean[swift_clean$bird_id == ab, ]
      bd$doy  <- as.integer(format(bd$timestamp, "%j"))
      bd      <- bd[order(bd$doy), ]
      bd_near <- bd[abs(bd$doy - cur_doy) == min(abs(bd$doy - cur_doy)), ]
      g <- g +
        geom_line(data = bd, aes(doy, lat, colour = colony, group = 1),
                  linewidth = 2.2, alpha = 1) +
        geom_point(data = bd_near, aes(doy, lat), size = 5.5, shape = 21,
                   fill = "#FFD700", colour = "#2a2a2a", stroke = 2, inherit.aes = FALSE)
    }
    g +
      scale_colour_manual(values = colony_colors) +
      scale_x_continuous(breaks = c(1,32,60,91,121,152,182,213,244,274,305,335),
                         labels = month.abb, limits = c(1, 365)) +
      scale_y_continuous(breaks = seq(-20, 60, 10),
                         labels = function(x) paste0(x, "°N")) +
      coord_cartesian(xlim = zoom_cycle$x, ylim = if (!is.null(zoom_cycle$y)) zoom_cycle$y else if (!is.null(ylim)) ylim else c(-20, 55)) +
      theme_swift() +
      labs(x = NULL, y = "Latitude", colour = NULL,
           title = if (!is.null(ab)) paste0("Annual Cycle — ", ab)
                   else "Annual Migration Cycle",
           subtitle = "Loess-smoothed mean latitude by day-of-year") +
      theme(legend.position = "bottom", legend.text = element_text(size = 6.5))
  }, res = 110, bg = "#1e2130")
}

In [34]:
shinyApp(ui, server)

Warning in scale_x_date(limits = ts + c(-60, 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = ts + c(-60, 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = ts + c(-60, 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = ts + c(-60, 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = ts + c(-60, 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = ts + c(-60, 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <


Listening on http://127.0.0.1:7456
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be t